# 02 — Preprocessing
### Topic Modelling on India News Headlines (2001–2023)

**Goal of this notebook:** turn the stratified sample from notebook 01 into text that's ready
for BERTopic modelling.

**Why preprocessing is different for BERTopic vs classical models (LDA/NMF):**
- Classical bag-of-words models (LDA/NMF) need heavy cleaning — stopwords, punctuation, and
  casing are all pure noise to them, since they only see word co-occurrence counts.
- BERTopic's **embedding step** uses a Transformer that was trained on natural sentences —
  punctuation, casing, and word order all carry meaning to it. Over-cleaning before embedding
  can *reduce* embedding quality.
- BERTopic's **topic representation step** (c-TF-IDF, run *after* clustering) is a bag-of-words
  method — this is where stopword removal and normalization genuinely help.

**So we keep two parallel columns:**
1. `text_for_embedding` — lightly cleaned → fed into the sentence embedding model.
2. `text_for_representation` — heavily cleaned/lemmatized → used only for the c-TF-IDF
   vectorizer inside BERTopic, and for coherence evaluation later.


In [ ]:
# Mount Drive first — this is the same persistent storage used in notebook 01, so we can
# read the sample it produced.
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/topic-modelling-capstone'
DATA_PROCESSED = f'{BASE_DIR}/data/processed'
os.makedirs(DATA_PROCESSED, exist_ok=True)

# --- Libraries ---
# contractions: expands "won't" -> "will not" etc, for consistency.
# nltk: stopword list, WordNet lemmatizer.
!pip install -q nltk contractions

import pandas as pd
import numpy as np
import re
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download the NLTK data files these tools rely on (quiet=True suppresses the download log).
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load the sampled dataset from notebook 01

In [ ]:
# Reads the exact file notebook 01 saved — same Drive path, so this works as long as
# notebook 01 has been run at least once.
df = pd.read_csv(f'{DATA_PROCESSED}/headlines_sample_300k.csv')
print(f"Loaded {len(df):,} rows")
df.head()


## 2. Deduplicate

News agencies sometimes republish near-identical headlines (wire copy, syndication). Exact
duplicates add no topical information and can bias cluster sizes, so we drop them here.


In [ ]:
before = len(df)
# drop_duplicates on headline_text specifically (rather than the whole row) catches headlines
# republished under a different date/category too, not just byte-identical rows.
df = df.drop_duplicates(subset='headline_text').reset_index(drop=True)
print(f"Dropped {before - len(df):,} exact duplicate headlines ({(before-len(df))/before:.2%})")
print(f"Remaining: {len(df):,} rows")


## 3. Light cleaning for embeddings (`text_for_embedding`)

Minimal intervention: fix encoding artifacts, expand contractions, normalize whitespace. We
deliberately **keep** casing and punctuation, since the embedding model uses them as signal.


In [ ]:
def light_clean(text):
    text = str(text)
    # Occasionally scraped/exported text has leftover HTML entities (&amp; etc) instead of
    # the actual characters — strip these out.
    text = re.sub(r'&amp;|&quot;|&#39;', ' ', text)
    # contractions.fix() expands things like "won't" -> "will not", "it's" -> "it is".
    # Wrapped in try/except because a handful of malformed strings can occasionally trip it up;
    # if that happens we just keep the text as-is rather than losing the row.
    try:
        text = contractions.fix(text)
    except Exception:
        pass
    # Collapse any run of whitespace (multiple spaces, tabs) into a single space, and trim
    # leading/trailing whitespace.
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# .apply() runs light_clean() on every row of headline_text and stores the result in a new column.
df['text_for_embedding'] = df['headline_text'].apply(light_clean)

# Spot check a few before/after examples to make sure the cleaning looks sensible.
df[['headline_text', 'text_for_embedding']].sample(5, random_state=RANDOM_STATE)


## 4. Heavy cleaning for topic representation (`text_for_representation`)

Lowercase, strip punctuation/digits, remove stopwords, lemmatize. This version is **only** used
for c-TF-IDF topic keyword extraction and for computing coherence scores — never fed to the
embedding model.


In [ ]:
# Standard English stopword list from NLTK (the, and, of, is, ...).
stop_words = set(stopwords.words('english'))

# A few extra near-stopwords specific to news headlines — these appear constantly but carry
# no topical meaning of their own (e.g. "said" attributes a quote but doesn't describe a topic).
custom_stopwords = {'said', 'says', 'say', 'today', 'yesterday', 'pm', 'am', 'mr', 'mrs',
                     'news', 'report', 'reports', 'reported'}
stop_words = stop_words.union(custom_stopwords)

# Lemmatizer reduces words to their dictionary base form (e.g. "elections" -> "election",
# "running" -> "running"... verb lemmatization needs a POS tag to fully normalize "running" ->
# "run"; we keep it simple here with default noun-based lemmatization, which is standard for
# topic modelling use cases).
lemmatizer = WordNetLemmatizer()

def heavy_clean(text):
    text = str(text).lower()
    # Replace anything that isn't a lowercase letter or whitespace with a space — this strips
    # punctuation, digits, and any stray symbols in one step.
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    # Keep a token only if: it's not a stopword, AND it's longer than 2 characters (drops
    # leftover single/double-letter noise from the regex step above, e.g. stray "s" from "'s").
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['text_for_representation'] = df['text_for_embedding'].apply(heavy_clean)

df[['text_for_embedding', 'text_for_representation']].sample(5, random_state=RANDOM_STATE)


## 5. Filter out low-signal documents

After cleaning, some headlines may be left with very few tokens (e.g. originally just names or
dates), which contribute noise rather than topical signal. We drop documents with fewer than
3 tokens in the representation column.


In [ ]:
# Count tokens remaining after heavy cleaning, per row.
df['token_count'] = df['text_for_representation'].str.split().apply(len)

before = len(df)
# Keep rows where: the lightly-cleaned text isn't empty/whitespace-only, AND there are at
# least 3 meaningful tokens left after heavy cleaning. Both conditions guard against feeding
# near-empty documents into the embedding/clustering steps later, which tend to just add noise.
df = df[
    (df['text_for_embedding'].str.strip().str.len() > 0) &
    (df['token_count'] >= 3)
].reset_index(drop=True)

print(f"Dropped {before - len(df):,} low-signal rows ({(before-len(df))/before:.2%})")
print(f"Final preprocessed dataset: {len(df):,} rows")

df['token_count'].describe()


## 6. Final sanity checks

Before saving, confirm the two text columns look reasonable and that we haven't accidentally
introduced empty strings or NaNs.


In [ ]:
print("Nulls in key columns:")
print(df[['text_for_embedding', 'text_for_representation']].isna().sum())

print("\nExample before/after, full pipeline:")
for i in df.sample(3, random_state=RANDOM_STATE).index:
    print(f"  original:       {df.loc[i, 'headline_text']}")
    print(f"  for embedding:  {df.loc[i, 'text_for_embedding']}")
    print(f"  for repr.:      {df.loc[i, 'text_for_representation']}")
    print()


## 7. Save the preprocessed dataset

This is what `03_modelling_bertopic.ipynb` will load directly.


In [ ]:
# Keep only the columns downstream notebooks actually need — no point carrying every
# intermediate column (like headline_word_count from notebook 01) forward.
cols_to_keep = ['publish_date', 'year', 'month', 'headline_category', 'category_top',
                'headline_text', 'text_for_embedding', 'text_for_representation', 'token_count']
df_out = df[[c for c in cols_to_keep if c in df.columns]]

df_out.to_csv(f'{DATA_PROCESSED}/headlines_preprocessed.csv', index=False)
print(f"Saved {len(df_out):,} preprocessed rows to {DATA_PROCESSED}/headlines_preprocessed.csv")


## 8. Summary of preprocessing decisions

*(Carry this into the report's "Procedure" section, filled in with your actual numbers.)*

- Started with `<fill in>` sampled rows from notebook 01
- Removed `<fill in>` exact duplicate headlines
- Removed `<fill in>` low-signal rows (fewer than 3 tokens after cleaning)
- Final preprocessed dataset: `<fill in>` rows
- Two parallel text representations kept: lightly-cleaned (`text_for_embedding`) for the
  Transformer embedding step, and heavily-cleaned/lemmatized (`text_for_representation`) for
  topic keyword extraction (c-TF-IDF) and coherence evaluation
- Rationale: over-cleaning before embedding degrades the semantic signal a Transformer relies on;
  under-cleaning before c-TF-IDF produces noisy, uninterpretable topic keyword lists. Splitting
  the two avoids that trade-off.
